# 02 - Editar medios propios y renderizar con Remotion en R2 privado

Guarda en Colab Secrets: `API_BASE_URL` (HTTPS publico), `WORKER_API_TOKEN`, `S3_ENDPOINT_URL`, `S3_BUCKET`, `S3_ACCESS_KEY_ID` y `S3_SECRET_ACCESS_KEY`. El worker descarga en privado los medios seleccionados desde R2, compone imagenes/videos propios, guion y voz, y sube el resultado. Ejecuta este notebook en una sesion de Colab separada del worker de investigacion.

**Modo worker continuo:** ejecuta las celdas desde arriba y deja activa la celda final. El dashboard local encola automáticamente los trabajos; esta sesión los toma mientras Colab permanezca conectado. Usa el botón **Detener** de Colab para desconectar el worker. Si la sesión gratuita se interrumpe, vuelve a abrir el notebook y ejecuta sus celdas otra vez.


In [ ]:
!apt-get update -qq && apt-get install -y -qq nodejs npm ffmpeg chromium

In [ ]:
%pip install -q boto3 requests

In [ ]:
import os

try:
    from google.colab import userdata
except ImportError:  # Ejecucion local o en CI: los secretos vienen del entorno.
    userdata = None


def _read_secret(name):
    """Lee un secreto de Colab y, si no estamos en Colab, de las variables de entorno."""
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            return value
    return os.environ.get(name)

REQUIRED = ("API_BASE_URL", "WORKER_API_TOKEN", "S3_ENDPOINT_URL", "S3_BUCKET",
            "S3_ACCESS_KEY_ID", "S3_SECRET_ACCESS_KEY")
for name in REQUIRED:
    value = _read_secret(name)
    if not value:
        raise RuntimeError(
            f"Configura el secreto {name} en Colab o exporta la variable de entorno {name}"
        )
    os.environ[name] = value
os.environ["API_BASE_URL"] = os.environ["API_BASE_URL"].rstrip("/")
if not os.environ["API_BASE_URL"].startswith("https://"):
    raise RuntimeError("API_BASE_URL debe ser HTTPS y accesible desde Colab")
API_HEADERS = {"Authorization": f"Bearer {os.environ['WORKER_API_TOKEN']}"}

In [ ]:
import json
import math
import shutil
import threading
import time
import uuid
import subprocess
from pathlib import Path

import boto3
import requests

WORKER_ID = uuid.uuid4().hex
WORKER_TYPE = "render"


PROJECT = Path("/content/marketing-content-os-renderer")
PUBLIC = PROJECT / "public"
SRC = PROJECT / "src"
PUBLIC.mkdir(parents=True, exist_ok=True)
SRC.mkdir(parents=True, exist_ok=True)
(PROJECT / "package.json").write_text(json.dumps({
    "private": True,
    "scripts": {"render": "remotion render src/index.ts MarketingShort"},
    "dependencies": {"@remotion/cli": "4.0.448", "react": "19.0.0", "react-dom": "19.0.0", "remotion": "4.0.448"},
    "devDependencies": {"@types/react": "19.0.0", "typescript": "5.7.3"}
}, indent=2), encoding="utf-8")
(SRC / "index.ts").write_text('import { registerRoot } from "remotion"; import { Root } from "./Root"; registerRoot(Root);', encoding="utf-8")
(SRC / "Root.tsx").write_text('''import React from "react";
import { AbsoluteFill, Audio, Composition, Img, OffthreadVideo, staticFile, interpolate, useCurrentFrame, useVideoConfig } from "remotion";
type Props = { topic: string; script: string; duration_seconds: number; audio_duration_seconds: number; visual_asset?: { asset_type: "image" | "video"; filename: string } | null };
const captions = (text: string) => {
  const words = text.trim().split(/\\s+/).filter(Boolean);
  const chunks: string[] = [];
  for (let index = 0; index < words.length; index += 7) chunks.push(words.slice(index, index + 7).join(" "));
  return chunks.length ? chunks : [""];
};
const Video: React.FC<Props> = ({ topic, script, audio_duration_seconds, visual_asset }) => {
  const frame = useCurrentFrame();
  const { fps } = useVideoConfig();
  const lines = captions(script);
  const segmentFrames = Math.max(1, audio_duration_seconds * fps / lines.length);
  const current = Math.min(lines.length - 1, Math.floor(frame / segmentFrames));
  const localFrame = frame - current * segmentFrames;
  const opacity = Math.min(1, Math.max(0.2, localFrame / 8));
  const offsetY = Math.max(0, (10 - localFrame) * 1.8);
  const zoom = interpolate(frame, [0, Math.max(1, audio_duration_seconds * fps)], [1, 1.045], { extrapolateRight: "clamp" });
  const progress = Math.min(1, frame / Math.max(1, audio_duration_seconds * fps));
  const mediaStyle: React.CSSProperties = { position: "absolute", inset: 0, width: "100%", height: "100%", objectFit: "cover", transform: "scale(" + zoom + ")" };
  return <AbsoluteFill style={{ background: "linear-gradient(145deg,#08121f,#16415d)", color: "white", fontFamily: "Arial, sans-serif" }}>
    {visual_asset?.asset_type === "image" && <Img src={staticFile(visual_asset.filename)} style={mediaStyle} />}
    {visual_asset?.asset_type === "video" && <OffthreadVideo src={staticFile(visual_asset.filename)} muted style={mediaStyle} />}
    <AbsoluteFill style={{ background: "linear-gradient(180deg,rgba(4,10,18,.52),rgba(4,10,18,.15) 42%,rgba(4,10,18,.68))" }} />
    <AbsoluteFill style={{ padding: 76, display: "flex", flexDirection: "column", justifyContent: "space-between" }}>
      <div>
        <div style={{ display: "inline-block", padding: "12px 20px", borderRadius: 999, background: "rgba(255,255,255,.16)", border: "1px solid rgba(255,255,255,.32)", fontSize: 24, letterSpacing: 3, fontWeight: 700 }}>IDEA CLAVE</div>
        <div style={{ marginTop: 28, fontSize: 54, lineHeight: 1.12, fontWeight: 800, textShadow: "0 3px 18px rgba(0,0,0,.55)" }}>{topic}</div>
      </div>
      <div style={{ opacity, transform: "translateY(" + offsetY + "px)", alignSelf: "center", maxWidth: "100%", padding: "26px 30px", borderRadius: 26, background: "rgba(3,9,16,.72)", border: "1px solid rgba(255,255,255,.24)", boxShadow: "0 16px 48px rgba(0,0,0,.28)", backdropFilter: "blur(8px)", fontSize: 48, lineHeight: 1.25, fontWeight: 700, textAlign: "center", textShadow: "0 2px 8px #000" }}>{lines[current]}</div>
      <div style={{ display: "flex", alignItems: "center", justifyContent: "space-between", fontSize: 24, letterSpacing: 2, fontWeight: 700 }}>
        <span>MARKETING CONTENT OS</span><span>{String(current + 1).padStart(2, "0")} / {String(lines.length).padStart(2, "0")}</span>
      </div>
      <div style={{ position: "absolute", left: 76, right: 76, bottom: 22, height: 6, background: "rgba(255,255,255,.28)", borderRadius: 99 }}>
        <div style={{ width: "100%", height: "100%", transform: "scaleX(" + progress + ")", transformOrigin: "left", background: "#58dcff", borderRadius: 99 }} />
      </div>
    </AbsoluteFill>
    <Audio src={staticFile("voice.wav")} />
  </AbsoluteFill>;
};
export const Root: React.FC = () => <Composition id="MarketingShort" component={Video} width={1080} height={1920} fps={30} durationInFrames={1050} defaultProps={{topic:"Tema",script:"",duration_seconds:35,audio_duration_seconds:35}} calculateMetadata={({props}) => ({durationInFrames:Math.max(30,Math.ceil(props.audio_duration_seconds*30))})} />;''', encoding="utf-8")
subprocess.run(["npm", "install", "--no-audit", "--no-fund"], cwd=PROJECT, check=True)
subprocess.run(["npx", "remotion", "browser", "ensure"], cwd=PROJECT, check=True)

In [ ]:
ACTIVE_STAGE = "idle"


def report_worker_status(current_job_id=None, stage="idle", stopped=False):
    global ACTIVE_STAGE
    ACTIVE_STAGE = "stopped" if stopped else stage
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/workers/heartbeat",
        headers=API_HEADERS,
        json={
            "worker_id": WORKER_ID,
            "worker": WORKER_TYPE,
            "current_job_id": str(current_job_id) if current_job_id else None,
            "stage": ACTIVE_STAGE,
            "stopped": stopped,
        },
        timeout=20,
    )
    response.raise_for_status()


def start_lease_heartbeat(job_id, lease_headers):
    stop = threading.Event()
    lease_lost = threading.Event()

    def renew_lease():
        while not stop.wait(60):
            try:
                response = requests.post(
                    f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/heartbeat",
                    headers=lease_headers,
                    timeout=30,
                )
                response.raise_for_status()
            except requests.RequestException:
                lease_lost.set()
                print(f"No se pudo renovar el lease del trabajo {job_id}; se detendra en el siguiente punto seguro.")
                return
            try:
                report_worker_status(job_id, ACTIVE_STAGE)
            except requests.RequestException as error:
                print(f"No se pudo actualizar la presencia del worker: {error}")

    thread = threading.Thread(target=renew_lease, daemon=True)
    thread.start()
    return stop, thread, lease_lost


def run_worker_loop(process_one, label, poll_seconds=20):
    print(f"Worker {label} conectado. La celda seguirá esperando trabajos; usa Detener para desconectarlo.")
    last_wait_notice = 0.0
    while True:
        try:
            report_worker_status(None, "idle")
            processed = process_one()
            if processed:
                report_worker_status(None, "idle")
                last_wait_notice = 0.0
                continue
            now = time.monotonic()
            if now - last_wait_notice >= 300:
                print(f"Worker {label} activo; sin trabajos pendientes. Reintentando cada {poll_seconds} s.")
                last_wait_notice = now
            time.sleep(poll_seconds)
        except KeyboardInterrupt:
            try:
                report_worker_status(None, "stopped", stopped=True)
            except requests.RequestException:
                pass
            print(f"Worker {label} detenido. Puedes volver a ejecutar esta celda para reconectarlo.")
            break
        except Exception as error:
            print(f"Error del worker {label}: {error}. Se reintentará en 30 s.")
            try:
                report_worker_status(None, "idle")
            except requests.RequestException:
                pass
            time.sleep(30)


def ensure_active_lease(lease_lost):
    if lease_lost.is_set():
        raise RuntimeError("El lease del trabajo se perdió; no se enviará el resultado tardío.")


def report_progress(job_id, lease_headers, stage, message, percent):
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/progress",
        headers=lease_headers,
        json={"stage": stage, "message": message, "percent": percent},
        timeout=30,
    )
    response.raise_for_status()
    try:
        report_worker_status(job_id, stage)
    except requests.RequestException as error:
        print(f"No se pudo actualizar la etapa del worker: {error}")


def process_one_render_job():
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/jobs/next",
        params={"worker": "render"}, headers=API_HEADERS, timeout=30,
    )
    response.raise_for_status()
    job = response.json()
    if job is None:
        return False

    job_id = job["job_id"]
    try:
        report_worker_status(job_id, "starting")
    except requests.RequestException as error:
        print(f"No se pudo actualizar la presencia del worker: {error}")
    lease_headers = {**API_HEADERS, "X-Worker-Lease": job["lease_token"]}
    heartbeat_stop, heartbeat_thread, lease_lost = start_lease_heartbeat(job_id, lease_headers)
    try:
        request = job["request"]
        report_progress(job_id, lease_headers, "media_download", "Descargando audio y medios privados de R2", 15)
        if not request.get("script") or not (request.get("audio_object_key") or request.get("audio_url")):
            raise ValueError("El trabajo requiere guion y audio")
        storage = boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT_URL"], region_name="auto",
                               aws_access_key_id=os.environ["S3_ACCESS_KEY_ID"],
                               aws_secret_access_key=os.environ["S3_SECRET_ACCESS_KEY"])
        if request.get("audio_object_key"):
            storage.download_file(os.environ["S3_BUCKET"], request["audio_object_key"], str(PUBLIC / "voice.wav"))
        else:
            audio = requests.get(request["audio_url"], timeout=120)
            audio.raise_for_status()
            (PUBLIC / "voice.wav").write_bytes(audio.content)
        duration_probe = subprocess.run([
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1", str(PUBLIC / "voice.wav"),
        ], capture_output=True, text=True, check=True)
        audio_duration_seconds = float(duration_probe.stdout.strip())
        if not math.isfinite(audio_duration_seconds) or audio_duration_seconds <= 0:
            raise ValueError("No se pudo determinar la duración de la narración")
        visual_asset = request.get("visual_asset")
        visual_props = None
        if visual_asset:
            suffix = {"image/png": ".png", "image/jpeg": ".jpg", "image/webp": ".webp", "video/mp4": ".mp4"}.get(visual_asset["content_type"])
            if suffix is None:
                raise ValueError("Formato de imagen/video no compatible para renderizar")
            visual_filename = f"background{suffix}"
            storage.download_file(os.environ["S3_BUCKET"], visual_asset["object_key"], str(PUBLIC / visual_filename))
            visual_props = {"asset_type": visual_asset["asset_type"], "filename": visual_filename}
        report_progress(job_id, lease_headers, "media_ready", "Medios listos; preparando edición", 35)
        props = {"topic": request["topic"], "script": request["script"],
                 "duration_seconds": request["duration_seconds"],
                 "audio_duration_seconds": audio_duration_seconds, "visual_asset": visual_props}
        props_path = Path("/content/render-props.json")
        props_path.write_text(json.dumps(props, ensure_ascii=False), encoding="utf-8")
        video_path = Path(f"/content/{job_id}.mp4")
        ensure_active_lease(lease_lost)
        report_progress(job_id, lease_headers, "rendering", "Editando y renderizando el video", 50)
        subprocess.run([
            "npx", "remotion", "render", "src/index.ts", "MarketingShort", str(video_path),
            f"--props={props_path}", "--codec=h264", "--audio-codec=aac", "--crf=18", "--x264-preset=veryfast",
            "--pixel-format=yuv420p", "--concurrency=2", "--browser-executable=/usr/bin/chromium",
        ], cwd=PROJECT, check=True)
        if not video_path.is_file() or video_path.stat().st_size == 0:
            raise RuntimeError("Remotion no produjo un MP4 valido")
        probe = subprocess.run([
            "ffprobe", "-v", "error", "-show_entries",
            "format=duration:stream=codec_type,codec_name,width,height",
            "-of", "json", str(video_path),
        ], capture_output=True, text=True, check=True)
        media_info = json.loads(probe.stdout)
        video_stream = next((stream for stream in media_info.get("streams", [])
                             if stream.get("codec_type") == "video"), None)
        audio_stream = next((stream for stream in media_info.get("streams", [])
                             if stream.get("codec_type") == "audio"), None)
        rendered_duration = float(media_info.get("format", {}).get("duration", 0))
        if not video_stream or video_stream.get("codec_name") != "h264":
            raise RuntimeError("El MP4 no contiene una pista de video H.264")
        if (video_stream.get("width"), video_stream.get("height")) != (1080, 1920):
            raise RuntimeError("El MP4 no tiene la resolución vertical 1080x1920")
        if not audio_stream or audio_stream.get("codec_name") != "aac":
            raise RuntimeError("El MP4 no contiene audio AAC")
        if rendered_duration <= 0 or abs(rendered_duration - audio_duration_seconds) > 1:
            raise RuntimeError("La duración del MP4 no coincide con la narración")

        report_progress(job_id, lease_headers, "video_storage", "Subiendo video terminado a R2 privado", 85)
        ensure_active_lease(lease_lost)
        video_key = f"renders/{job_id}.mp4"
        storage.upload_file(str(video_path), os.environ["S3_BUCKET"], video_key,
                            ExtraArgs={"ContentType": "video/mp4"})
        ensure_active_lease(lease_lost)
        complete = requests.post(
            f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/complete",
            headers=lease_headers, json={"output": {"video_object_key": video_key}}, timeout=60,
        )
        complete.raise_for_status()
        print(f"Video listo: {job_id}")
    except Exception as error:
        try:
            failed = requests.post(
                f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/fail",
                headers=lease_headers, json={"error": str(error)[:1900]}, timeout=60,
            )
            failed.raise_for_status()
        except requests.RequestException:
            print("No se pudo informar el fallo; la API recuperará el trabajo al vencer el lease.")
        raise RuntimeError(f"Trabajo {job_id} no completado: {error}") from error
    finally:
        heartbeat_stop.set()
        heartbeat_thread.join(timeout=5)
    return True


# Mantén esta celda ejecutándose: el dashboard local encola los trabajos y este worker
# los procesa en orden. Detén la celda cuando quieras desconectar esta sesión de Colab.
run_worker_loop(process_one_render_job, "render de video")
